**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# RMSE Comparison with Significance — DL Noise-Aware vs DL Noise-Free vs DM
Loads pre-generated noisy dicts, verifies model performance, runs Wilcoxon tests,
and adds significance stars to the RMSE figure.

In [ ]:
import os, json
import numpy as np
import scipy.io as sio
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
CONFIG = {
    'param_path'        : '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'noisefree_sig_path': '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'dict_base_path'    : '../subsamples/subsamples_v3',
    'echotimes_path'    : '../echotimes.mat',
    'nf_model_ckpt'     : './results/triple_regime_nf_results_v1/models/triple_nf_best.pt',
    'noisy_model_ckpt'  : './results/triple_regime_results_v1/models/triple_regime_best.pt',
    'output_dir'        : './results/statistical_comparison',
    'nf_rmse_json'      : './results/triple_regime_nf_results_v1/per_snr_rmse.json',
    'noisy_rmse_json'   : './results/triple_regime_results_v1/per_snr_rmse.json',
    'dict_key'          : 'Dico40_save',
    'noisy_dict_key'    : 'Dico40_noisy',
    'param_key'         : 'par_save',
    'param_mins': np.array([0.0,   0.0025,  1.0e-6,  0.050]),
    'param_maxs': np.array([1.0,   0.15,   25.0e-6,  0.200]),
    'param_names': ['SO2', 'CBV', 'R', 'T2'],
    'n_fid'   : 14,
    'n_rephas': 16,
    'n_postse': 10,
    'R2starA_min':  2.0,  'R2starA_max': 55.0,
    'R2starB_min': -30.0, 'R2starB_max': 22.0,
    'R2starC_min':  2.0,  'R2starC_max': 55.0,
    'test_frac'        : 0.15,
    'snr_levels'       : [20, 50, 100, 150],
    'max_test_samples' : None,  # None = all; set 50000 for speed
}
SE_ECHO     = CONFIG['n_fid'] + CONFIG['n_rephas']
PKEYS       = CONFIG['param_names']
PARAM_NAMES = ['SO\u2082', 'CBV', 'R', 'T2']
PARAM_UNITS = ['(%)', '(%)', '(\u03bcm)', '(ms)']
PARAM_SCALE = [100, 100, 1e6, 1000]
PHYS        = np.array(PARAM_SCALE, dtype=np.float32)
N_TESTS     = len(PKEYS) * len(CONFIG['snr_levels'])
C_DM        = '#E65100'
C_DL_NF     = '#AD1457'
C_DL_NOISY  = '#00695C'
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f'Bonferroni N_TESTS = {N_TESTS}')

In [ ]:
def load_mat_nf(path, key):
    """Load noise-free dictionary — shape (N, 40), rows = samples."""
    try:
        mat = sio.loadmat(path)
        arr = np.array(mat[key] if key in mat else mat[[k for k in mat if not k.startswith('_')][0]],
                       dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            arr = np.array(f[key] if key in f else f[list(f.keys())[0]], dtype=np.float32)
            if arr.ndim >= 2: arr = arr.T
    if arr.shape[0] == 40 and arr.shape[1] != 40:
        arr = arr.T
    return arr

def load_mat_noisy(path, key='Dico40_noisy'):
    """Load noisy dictionary — HDF5, shape (40, N) -> transpose to (N, 40)."""
    try:
        mat = sio.loadmat(path)
        arr = np.array(mat[key] if key in mat else mat[[k for k in mat if not k.startswith('_')][0]],
                       dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            arr = np.array(f[key] if key in f else f[list(f.keys())[0]], dtype=np.float32)
    if arr.shape[0] == 40 and arr.shape[1] != 40:
        arr = arr.T
    print(f'  Loaded noisy dict: {arr.shape}, range [{arr.min():.4f}, {arr.max():.4f}]')
    return arr

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    return signals[valid], params[valid]

def compute_triple_features(signals, cfg):
    et  = sio.loadmat(cfg['echotimes_path'])['Echotimes'].flatten() / 1000.0
    tA  = et[:cfg['n_fid']]
    tB  = et[cfg['n_fid']:SE_ECHO]
    tC  = et[SE_ECHO:] - et[SE_ECHO - 1]
    def ols(t, s):
        logs = np.log(np.maximum(np.abs(s), 1e-12))
        A = np.column_stack([t, np.ones_like(t)])
        return -np.linalg.lstsq(A, logs.T, rcond=None)[0][0]
    return (ols(tA, signals[:, :cfg['n_fid']]).astype(np.float32),
            ols(tB, signals[:, cfg['n_fid']:SE_ECHO]).astype(np.float32),
            ols(tC, signals[:, SE_ECHO:]).astype(np.float32))

def scale_features(R2A, R2B, R2C, cfg):
    fA = np.clip((R2A - cfg['R2starA_min']) / (cfg['R2starA_max'] - cfg['R2starA_min']), 0, 1)
    fB = np.clip((R2B - cfg['R2starB_min']) / (cfg['R2starB_max'] - cfg['R2starB_min']), 0, 1)
    fC = np.clip((R2C - cfg['R2starC_min']) / (cfg['R2starC_max'] - cfg['R2starC_min']), 0, 1)
    return fA, fB, fC

def build_input(signals, cfg):
    sig_n = euclidean_norm(signals)
    R2A, R2B, R2C = compute_triple_features(signals, cfg)
    fA, fB, fC = scale_features(R2A, R2B, R2C, cfg)
    X = np.column_stack([sig_n, fA[:,None], fB[:,None], fC[:,None]]).astype(np.float32)
    valid = np.all(np.isfinite(X), axis=1)
    return X, valid

def predict(model, x_np, batch=8192):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch):
            xb = torch.tensor(x_np[i:i+batch], dtype=torch.float32).to(device)
            out.append(model(xb).cpu().numpy())
    return np.concatenate(out, axis=0)

def dm_gpu(dict_norm, dict_params, test_norm, vox_chunk=256, dict_chunk=100_000):
    """GPU-accelerated dictionary matching chunked to fit 6 GB VRAM."""
    best_idx   = np.zeros(len(test_norm), dtype=np.int32)
    best_score = np.full(len(test_norm), -np.inf, dtype=np.float32)
    with torch.no_grad():
        for d0 in range(0, len(dict_norm), dict_chunk):
            d1     = min(d0 + dict_chunk, len(dict_norm))
            dict_t = torch.tensor(dict_norm[d0:d1], dtype=torch.float32).to(device)
            for v0 in range(0, len(test_norm), vox_chunk):
                v1    = min(v0 + vox_chunk, len(test_norm))
                vox_t = torch.tensor(test_norm[v0:v1], dtype=torch.float32).to(device)
                sc, idx = torch.mm(vox_t, dict_t.T).max(dim=1)
                sc = sc.cpu().numpy(); idx = idx.cpu().numpy() + d0
                better = sc > best_score[v0:v1]
                best_score[v0:v1][better] = sc[better]
                best_idx[v0:v1][better]   = idx[better]
            del dict_t; torch.cuda.empty_cache()
    return dict_params[best_idx]

print('Utilities ready')

In [ ]:
class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, n_features=3, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, feature_dim * 2)
        )
    def forward(self, x, regime):
        g, b = self.net(regime).chunk(2, dim=-1)
        return x * (1 + g) + b

class TripleRegimeModel(nn.Module):
    def __init__(self, n_outputs=4, dropout=0.05):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,32,7,padding=3),    nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32,64,5,padding=2),   nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64,128,3,padding=1),  nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256,3,padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256,256,3,padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.fc1 = nn.Linear(1280,512); self.bn1 = nn.BatchNorm1d(512); self.film1 = FiLMLayer(512)
        self.fc2 = nn.Linear(512,256);  self.bn2 = nn.BatchNorm1d(256); self.film2 = FiLMLayer(256)
        self.fc3 = nn.Linear(256,128);  self.bn3 = nn.BatchNorm1d(128); self.film3 = FiLMLayer(128)
        self.fc_out = nn.Linear(128, n_outputs)
    def forward(self, x):
        sig = x[:,:40].unsqueeze(1); regime = x[:,40:]
        h = self.conv(sig).flatten(1)
        h = self.film1(torch.relu(self.bn1(self.fc1(h))), regime)
        h = self.film2(torch.relu(self.bn2(self.fc2(h))), regime)
        h = self.film3(torch.relu(self.bn3(self.fc3(h))), regime)
        return torch.sigmoid(self.fc_out(h))

model_nf = TripleRegimeModel().to(device)
model_nf.load_state_dict(torch.load(CONFIG['nf_model_ckpt'], map_location=device))
model_nf.eval(); print('NF model loaded')

model_noisy = TripleRegimeModel().to(device)
model_noisy.load_state_dict(torch.load(CONFIG['noisy_model_ckpt'], map_location=device))
model_noisy.eval(); print('Noisy model loaded')

In [ ]:
sig_nf  = load_mat_nf(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
par_raw = load_mat_nf(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]
sig_nf, par_raw = filter_param_range(sig_nf, par_raw, CONFIG['param_mins'], CONFIG['param_maxs'])
sig_nf, par_raw = clean_data(sig_nf, par_raw)
print(f'NF dict: {sig_nf.shape}')

_, idx_test = train_test_split(np.arange(len(sig_nf)), test_size=CONFIG['test_frac'], random_state=42)
dict_norm    = euclidean_norm(sig_nf)
par_phys_all = par_raw * PHYS
print(f'Test indices: {len(idx_test):,}   DM dict: {len(dict_norm):,}')

In [ ]:
# Check which checkpoint is actually best
import os

candidates = [
    './results/triple_regime_results_v1/models/triple_regime_best.pt',
    './results/triple_regime_results_v1/models/triple_regime_final.pt',
    './results/triple_regime_results_v1/models/triple_best.pt',
]

for path in candidates:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'EXISTS  {path}  ({size_mb:.1f} MB)')
    else:
        print(f'MISSING {path}')

In [ ]:
hist_path = './results/triple_regime_results_v1/training_history.json'
if os.path.exists(hist_path):
    with open(hist_path) as f: hist = json.load(f)
    print('Final train loss:', hist['train_loss'][-1])
    print('Final val loss:  ', hist['val_loss'][-1])
    print('Best val loss:   ', min(hist['val_loss']))
    print('Best epoch:      ', hist['val_loss'].index(min(hist['val_loss'])) + 1)

In [ ]:
# Sanity check: run noisy model on NF signals — should match original per_snr_rmse.json at high SNR
mid = np.where(
    (par_raw[:, 0] > 0.5) & (par_raw[:, 0] < 0.7) &
    (par_raw[:, 1] > 0.02) & (par_raw[:, 1] < 0.05)
)[0][:10]

X_chk, valid_chk = build_input(sig_nf[mid], CONFIG)
pred_chk = predict(model_noisy, X_chk[valid_chk])
pred_phys_chk = params_inverse(pred_chk, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4]) * PHYS
true_phys_chk = par_raw[mid][valid_chk] * PHYS

print('Noise-aware model on NF signals:')
print('Predicted:');     print(pd.DataFrame(pred_phys_chk, columns=PKEYS).round(2))
print('Ground truth:');  print(pd.DataFrame(true_phys_chk, columns=PKEYS).round(2))
rmse_chk = np.sqrt(np.mean((pred_phys_chk - true_phys_chk)**2, axis=0))
print('\nRMSE (should be close to SNR=150 values from original JSON):')
for p, r, u in zip(PKEYS, rmse_chk, PARAM_UNITS): print(f'  {p}: {r:.3f} {u}')

# Load original JSON for comparison
orig_noisy_json = None
if os.path.exists(CONFIG['noisy_rmse_json']):
    with open(CONFIG['noisy_rmse_json']) as f: orig_noisy_json = json.load(f)
    print('\nOriginal per_snr_rmse.json (SNR=150):')
    for p, sc, u in zip(PKEYS, PARAM_SCALE, PARAM_UNITS):
        print(f'  {p}: {float(orig_noisy_json[p][-1])*sc:.3f} {u}')

In [ ]:
all_errors   = {snr: {'DL_noisy': {}, 'DL_NF': {}, 'DM': {}} for snr in CONFIG['snr_levels']}
rmse_summary = {m: {p: [] for p in PKEYS} for m in ['DL_noisy', 'DL_NF', 'DM']}

for snr in CONFIG['snr_levels']:
    print(f'\n── SNR = {snr} ──────────────────────────')
    noisy_path    = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_noisy_all = load_mat_noisy(noisy_path, CONFIG['noisy_dict_key'])

    test_idx = idx_test.copy()
    max_n = CONFIG['max_test_samples']
    if max_n and len(test_idx) > max_n:
        test_idx = np.random.default_rng(0).choice(test_idx, max_n, replace=False)

    sig_test = sig_noisy_all[test_idx]
    par_eval = par_raw[test_idx] * PHYS

    X, valid  = build_input(sig_test, CONFIG)
    X = X[valid]; par_v = par_eval[valid]; sig_v = sig_test[valid]
    print(f'  Valid: {valid.sum():,}')

    p_noisy = params_inverse(predict(model_noisy, X), CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4]) * PHYS
    p_nf    = params_inverse(predict(model_nf,    X), CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4]) * PHYS
    p_dm    = dm_gpu(dict_norm, par_phys_all, euclidean_norm(sig_v))

    print(f'  {"":5} {"DL_noisy":>10} {"DL_NF":>10} {"DM":>10}')
    for j, (pkey, unit) in enumerate(zip(PKEYS, PARAM_UNITS)):
        t = par_v[:, j]
        all_errors[snr]['DL_noisy'][pkey] = np.abs(p_noisy[:, j] - t)
        all_errors[snr]['DL_NF'][pkey]    = np.abs(p_nf[:, j]    - t)
        all_errors[snr]['DM'][pkey]       = np.abs(p_dm[:, j]    - t)
        r_n = np.sqrt(np.mean(all_errors[snr]['DL_noisy'][pkey]**2))
        r_f = np.sqrt(np.mean(all_errors[snr]['DL_NF'][pkey]**2))
        r_d = np.sqrt(np.mean(all_errors[snr]['DM'][pkey]**2))
        rmse_summary['DL_noisy'][pkey].append(r_n)
        rmse_summary['DL_NF'][pkey].append(r_f)
        rmse_summary['DM'][pkey].append(r_d)
        print(f'  {pkey:<5} {r_n:>8.3f}{unit}  {r_f:>8.3f}{unit}  {r_d:>8.3f}{unit}')

print('\nDone.')

In [ ]:
orig_nf    = None; orig_noisy = None
if os.path.exists(CONFIG['nf_rmse_json']):
    with open(CONFIG['nf_rmse_json']) as f: orig_nf = json.load(f)
if os.path.exists(CONFIG['noisy_rmse_json']):
    with open(CONFIG['noisy_rmse_json']) as f: orig_noisy = json.load(f)

print(f'{"Param":<5} {"SNR":<6} {"NF_now":>9} {"NF_orig":>9} {"Noisy_now":>11} {"Noisy_orig":>11}')
print('-' * 58)
for pkey, sc in zip(PKEYS, PARAM_SCALE):
    for i, snr in enumerate(CONFIG['snr_levels']):
        nf_n  = rmse_summary['DL_NF'][pkey][i]
        ny_n  = rmse_summary['DL_noisy'][pkey][i]
        nf_o  = float(orig_nf[pkey][i])    * sc if orig_nf    else float('nan')
        ny_o  = float(orig_noisy[pkey][i]) * sc if orig_noisy else float('nan')
        match = 'OK' if abs(ny_n - ny_o) < ny_o * 0.05 else 'CHECK'
        print(f'{pkey:<5} {snr:<6} {nf_n:>9.3f} {nf_o:>9.3f} {ny_n:>11.3f} {ny_o:>11.3f}  {match}')

In [ ]:
sig_table = {comp: {p: [] for p in PKEYS}
             for comp in ['DL_noisy vs DM', 'DL_noisy vs DL_NF']}
rows = []

for comp, (mA, mB) in [('DL_noisy vs DM',    ('DL_noisy', 'DM')),
                        ('DL_noisy vs DL_NF', ('DL_noisy', 'DL_NF'))]:
    for snr in CONFIG['snr_levels']:
        for pkey, unit in zip(PKEYS, PARAM_UNITS):
            eA = all_errors[snr][mA][pkey]
            eB = all_errors[snr][mB][pkey]
            _, p_raw = stats.wilcoxon(eA, eB, alternative='two-sided')
            p_corr   = min(p_raw * N_TESTS, 1.0)
            sig = '***' if p_corr < 0.001 else ('**' if p_corr < 0.01 else ('*' if p_corr < 0.05 else 'ns'))
            sig_table[comp][pkey].append(sig)
            rows.append({'Comparison': comp, 'SNR': snr, 'Parameter': pkey,
                         'RMSE_A': round(np.sqrt(np.mean(eA**2)), 3),
                         'RMSE_B': round(np.sqrt(np.mean(eB**2)), 3),
                         'p_raw': f'{p_raw:.2e}', 'p_Bonferroni': f'{p_corr:.2e}',
                         'Significance': sig})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(CONFIG['output_dir'], 'statistical_comparison.csv'), index=False)
print(df.to_string(index=False))

In [ ]:
x_pos   = np.arange(len(CONFIG['snr_levels']))
xlabels = [str(s) for s in CONFIG['snr_levels']]
styles  = {
    'DM'      : dict(color=C_DM,       marker='D', ls=':',  lw=2, ms=6, label='DM (NF-dict)'),
    'DL_NF'   : dict(color=C_DL_NF,    marker='s', ls='--', lw=2, ms=6, label='DL Noise-free'),
    'DL_noisy': dict(color=C_DL_NOISY, marker='^', ls='-',  lw=2, ms=6, label='DL Noise-aware'),
}

fig, axes = plt.subplots(2, 2, figsize=(8, 7),
                          gridspec_kw={'hspace': 0.45, 'wspace': 0.38})
axes = axes.flatten()

for ax, pkey, pname, punit in zip(axes, PKEYS, PARAM_NAMES, PARAM_UNITS):
    for m in ['DM', 'DL_NF', 'DL_noisy']:
        ax.plot(x_pos, rmse_summary[m][pkey], **styles[m])
    ax.set_title(pname, fontsize=11, fontweight='bold')
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_xticks(x_pos); ax.set_xticklabels(xlabels, fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4); ax.set_axisbelow(True)

    # Stars: DL_noisy vs DM above each SNR point where significant
    y_max = max(max(rmse_summary['DM'][pkey]), max(rmse_summary['DL_NF'][pkey]))
    y_top = y_max * 1.10
    for xi, sig in enumerate(sig_table['DL_noisy vs DM'][pkey]):
        if sig != 'ns':
            ax.text(xi, y_top, sig, ha='center', va='bottom',
                    fontsize=10, fontweight='bold', color=C_DM)
    ax.set_ylim(top=y_top * 1.15)

axes[0].legend(fontsize=7.5, framealpha=0.9, loc='upper right')
fig.text(0.5, 0.01,
         'Stars: DL Noise-aware vs DM  (Wilcoxon, Bonferroni-corrected)  *p<0.05  **p<0.01  ***p<0.001',
         ha='center', fontsize=7.5, style='italic', color='#555')
fig.suptitle('Noise Robustness Analysis', fontsize=11, fontweight='bold')
plt.savefig(os.path.join(CONFIG['output_dir'], 'Fig_RMSE_with_stats.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: Fig_RMSE_with_stats.png')